# ⚡ AI Sports Performance Assistant — Full RAG Pipeline
### SJSU Graduate Final Project | Spring 2026
**Retrieval-Augmented Generation for Evidence-Based Athletic Guidance**

This notebook runs the **complete RAG pipeline** end-to-end:
1. Install dependencies
2. Upload project data (chunks, FAISS index, metadata)
3. Build retrieval module (FAISS semantic search)
4. Build generation module (Google Gemini 2.5 Flash)
5. Run all 10 benchmark evaluation queries
6. Compare RAG vs Baseline answers
7. Visualize evaluation results
8. Launch the full Streamlit app (optional)

> **Note:** The Streamlit web UI (with the readiness gauge) requires a tunnel to run from Colab. This notebook demonstrates the core RAG pipeline directly.

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu google-genai pandas numpy python-dotenv tqdm matplotlib

## 2. Set Your Google Gemini API Key
Get a free key at: https://aistudio.google.com/app/apikey

In [ ]:
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named GOOGLE_API_KEY
try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    # Option 2: Paste directly (less secure)
    os.environ['GOOGLE_API_KEY'] = 'PASTE_YOUR_KEY_HERE'  # <-- Replace this
    print('⚠️ Using hardcoded key — consider using Colab Secrets instead')

## 3. Upload Project Data
Upload the following files from your `sports_rag_project/` folder:
- `chunks.csv`
- `metadata.csv`
- `faiss_index/sports_rag.index`
- `evaluation_questions.csv`

In [ ]:
from google.colab import files
import os

os.makedirs('faiss_index', exist_ok=True)

print('📁 Upload chunks.csv, metadata.csv, evaluation_questions.csv')
print('   Then upload sports_rag.index (it will be moved to faiss_index/)')
uploaded = files.upload()

# Move FAISS index to correct directory
if 'sports_rag.index' in uploaded:
    os.rename('sports_rag.index', 'faiss_index/sports_rag.index')
    print('✅ FAISS index moved to faiss_index/')

# Verify
for f in ['chunks.csv', 'metadata.csv', 'faiss_index/sports_rag.index']:
    print(f"{'✅' if os.path.exists(f) else '❌'} {f}")

## 4. Explore the Knowledge Base

In [ ]:
import pandas as pd

chunks_df = pd.read_csv('chunks.csv')
metadata_df = pd.read_csv('metadata.csv')

print(f'📊 Total source documents: {len(metadata_df)}')
print(f'📊 Total text chunks: {len(chunks_df)}')
print(f'\n🏅 Sports covered: {chunks_df["sport"].unique().tolist()}')
print(f'📂 Categories: {chunks_df["category"].unique().tolist()}')
print(f'🏢 Organizations: {chunks_df["organization"].nunique()} unique orgs')
print()
print('── Chunks per Sport × Category ──')
print(chunks_df.groupby(['sport', 'category']).size().unstack(fill_value=0))
print()
print('── Sample Chunk ──')
print(chunks_df.iloc[0]['text'][:500])

## 5. Build the Retrieval Module (FAISS Semantic Search)
This is the core of the RAG pipeline — it finds the most relevant document chunks for any query.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model and FAISS index
print('Loading embedding model (all-MiniLM-L6-v2)...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index = faiss.read_index('faiss_index/sports_rag.index')
print(f'✅ Model loaded | FAISS index: {faiss_index.ntotal} vectors, {faiss_index.d} dimensions')


def retrieve(query, sport_filter=None, category_filter=None, top_k=5):
    """Retrieve top_k most relevant chunks for a query with optional filters."""
    query_vec = embed_model.encode([query]).astype('float32')
    search_k = top_k * 10 if (sport_filter or category_filter) else top_k
    distances, indices = faiss_index.search(query_vec, search_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        row = chunks_df.iloc[idx]
        if sport_filter and row['sport'].lower() != sport_filter.lower():
            continue
        if category_filter and row['category'].lower() != category_filter.lower():
            continue
        results.append({
            'chunk_id': row['chunk_id'], 'source_id': row['source_id'],
            'sport': row['sport'], 'category': row['category'],
            'title': row['title'], 'organization': row['organization'],
            'text': row['text'], 'score': float(dist)
        })
        if len(results) >= top_k:
            break
    return results

# Quick test
test = retrieve('How much protein for strength training?', 'Strength Training', 'Nutrition', 3)
for i, r in enumerate(test):
    print(f'\n--- Result {i+1} (L2={r["score"]:.3f}) ---')
    print(f'Source: {r["organization"]} | {r["sport"]} > {r["category"]}')
    print(f'Text: {r["text"][:200]}...')

## 6. Build the Generation Module (Gemini 2.5 Flash)
Two modes: **RAG** (with retrieved chunks) and **Baseline** (LLM-only, no retrieval).

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
MODEL_NAME = 'gemini-2.5-flash'

SYSTEM_PROMPT = (
    'You are an AI Sports Performance Assistant for beginner to intermediate athletes. '
    'Provide detailed, evidence-based guidance on training, nutrition, and recovery. '
    'Never diagnose injuries, prescribe supplements, or give medical advice. '
    'Always recommend consulting a qualified professional for medical needs. '
    'Be thorough, specific, and encouraging. Use numbers and specifics from the evidence.'
)

RAG_STRUCTURE = '''Respond using these exact sections. Be informative and specific, but concise:

**📋 Key Insight**
2-3 sentences with the core answer and most important specific number or protocol.

**🏋️ Recommendations**
4-5 bullet points with specific protocols (numbers, sets, reps, grams, minutes) and why each matters.

**🍽️ How to Implement**
3-4 actionable steps to fit this into the athlete\'s weekly routine.

**📅 Sample Protocol**
One concrete, ready-to-use example (workout day, meal plan, or weekly schedule).

**📚 Evidence Base**
1-2 sentences naming specific organizations and their key findings.

**⚠️ Important Limits**
1 sentence: what this does NOT cover and when to see a professional.'''


def call_gemini(prompt):
    """Call Gemini and return the full response text."""
    try:
        response = client.models.generate_content(
            model=MODEL_NAME, contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.25, max_output_tokens=2000
            )
        )
        return response.text
    except Exception as e:
        return f'⚠️ API Error: {e}'


def get_rag_answer(question, chunks, profile):
    """Generate a RAG answer grounded in retrieved chunks."""
    context = '\n\n'.join(
        f"[{c['organization']}, {c['sport']} {c['category']}]\n{c['text']}" for c in chunks
    )
    prompt = (
        f"Athlete Profile: Sport={profile['sport']} | Category={profile['category']} | "
        f"Goal={profile['goal']} | Level={profile['experience']} | "
        f"Training={profile['training_days']} days/wk | Weight={profile['body_weight']} kg\n"
        f"IMPORTANT DIRECTIVES:\n"
        f"1. Sport: {profile['sport']} only. No cross-sport recommendations.\n"
        f"2. Goal: '{profile['goal']}' — all advice must serve this goal.\n"
        f"3. Category: '{profile['category']}' — frame entire answer through this lens.\n\n"
        f"Question: {question}\n\nRetrieved Evidence:\n{'='*50}\n{context}\n{'='*50}\n\n"
        f"{RAG_STRUCTURE}\n\nUse evidence above. Be complete."
    )
    return call_gemini(prompt)


def get_baseline_answer(question, profile):
    """Generate a baseline LLM-only answer (no retrieval)."""
    prompt = (
        f"Athlete Profile: Sport={profile['sport']} | Category={profile['category']} | "
        f"Goal={profile['goal']} | Level={profile['experience']} | "
        f"Training={profile['training_days']} days/wk | Weight={profile['body_weight']} kg\n"
        f"IMPORTANT DIRECTIVES:\n"
        f"1. Sport: {profile['sport']} only.\n"
        f"2. Goal: '{profile['goal']}' — all advice must serve this goal.\n"
        f"3. Category: '{profile['category']}' — frame answer through this lens.\n\n"
        f"Question: {question}\n\n{RAG_STRUCTURE}\n\n"
        f"Answer from general sports science knowledge. Do not fabricate citations."
    )
    return call_gemini(prompt)

print('✅ Generation module ready')

## 7. Demo: Single Query — RAG vs Baseline

In [ ]:
from IPython.display import Markdown, display

# Configure query
QUESTION = 'What are effective interval training methods for a beginner soccer player?'
PROFILE = {
    'sport': 'Soccer', 'category': 'Training', 'goal': 'Improve Performance',
    'experience': 'Beginner', 'training_days': '3', 'body_weight': '70'
}

# Retrieve
chunks = retrieve(QUESTION, PROFILE['sport'], PROFILE['category'], top_k=6)
print(f'🔎 Retrieved {len(chunks)} chunks from: {", ".join(set(c["organization"] for c in chunks))}')

# Generate RAG answer
print('\n' + '='*80)
print('🔬 RAG ANSWER — EVIDENCE GROUNDED')
print('='*80)
rag_ans = get_rag_answer(QUESTION, chunks, PROFILE)
display(Markdown(rag_ans))

# Generate Baseline answer
print('\n' + '='*80)
print('💬 BASELINE LLM — NO RETRIEVED CONTEXT')
print('='*80)
base_ans = get_baseline_answer(QUESTION, PROFILE)
display(Markdown(base_ans))

## 8. Run Full 10-Question Evaluation
Runs all benchmark questions through both RAG and Baseline pipelines.

In [ ]:
import time

eval_df = pd.read_csv('evaluation_questions.csv')
results = []

for _, row in eval_df.iterrows():
    qid = row['question_id']
    print(f'\n⏳ {qid}: {row["question"][:60]}...')

    profile = {
        'sport': row['sport'], 'category': row['category'],
        'goal': 'Improve Performance', 'experience': row['experience_level'],
        'training_days': str(row['training_days']),
        'body_weight': str(row['body_weight_kg'])
    }

    # Retrieve
    chunks = retrieve(row['question'], row['sport'], row['category'], top_k=6)
    if not chunks:
        chunks = retrieve(row['question'], top_k=6)

    # Generate both answers
    rag_ans = get_rag_answer(row['question'], chunks, profile)
    time.sleep(2)  # Rate limiting
    base_ans = get_baseline_answer(row['question'], profile)
    time.sleep(2)

    precision = sum(1 for c in chunks if c['sport'].lower() == row['sport'].lower()) / len(chunks) if chunks else 0

    results.append({
        'question_id': qid, 'sport': row['sport'], 'category': row['category'],
        'question': row['question'], 'rag_answer': rag_ans, 'baseline_answer': base_ans,
        'chunks_retrieved': len(chunks), 'retrieval_precision': f'{precision:.0%}',
        'sources': ', '.join(set(c['organization'] for c in chunks))
    })
    print(f'  ✅ Done | {len(chunks)} chunks | Precision: {precision:.0%}')

results_df = pd.DataFrame(results)
results_df.to_csv('colab_evaluation_results.csv', index=False)
print(f'\n🎉 All {len(results_df)} evaluations complete! Saved to colab_evaluation_results.csv')

## 9. View Evaluation Results

In [ ]:
from IPython.display import Markdown, display

for _, r in results_df.iterrows():
    print(f'\n{"="*80}')
    print(f'📌 {r["question_id"]} | {r["sport"]} > {r["category"]} | Precision: {r["retrieval_precision"]}')
    print(f'   Q: {r["question"]}')
    print(f'   Sources: {r["sources"]}')
    print(f'{"="*80}')
    display(Markdown(f'### 🔬 RAG Answer\n{r["rag_answer"][:800]}...'))
    display(Markdown(f'### 💬 Baseline Answer\n{r["baseline_answer"][:800]}...'))

## 10. Retrieval Precision Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Chunks per sport/category
ct = chunks_df.groupby(['sport', 'category']).size().unstack(fill_value=0)
ct.plot(kind='bar', ax=axes[0], colormap='viridis', edgecolor='white')
axes[0].set_title('Knowledge Base: Chunks per Sport × Category', fontweight='bold')
axes[0].set_ylabel('Number of Chunks')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='Category')

# Chart 2: Retrieval precision per query
prec = results_df['retrieval_precision'].str.rstrip('%').astype(float)
colors = ['#4CAF50' if p >= 80 else '#FF9800' if p >= 60 else '#F44336' for p in prec]
axes[1].barh(results_df['question_id'], prec, color=colors, edgecolor='white')
axes[1].set_xlabel('Retrieval Precision (%)')
axes[1].set_title('Retrieval Precision@6 per Benchmark Query', fontweight='bold')
axes[1].set_xlim(0, 105)
axes[1].axvline(x=77, color='red', linestyle='--', alpha=0.5, label='Avg (77%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluation_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Charts saved to evaluation_charts.png')

## 11. (Optional) Launch the Full Streamlit App from Colab
This uses `localtunnel` to create a public URL for the Streamlit app.

> **Note:** You must first upload `app.py`, `retrieve.py`, `generate.py` for this to work.

In [ ]:
# Upload the remaining app files
from google.colab import files
print('📁 Upload: app.py, retrieve.py, generate.py')
uploaded = files.upload()

In [ ]:
# Create .env file for the app
with open('.env', 'w') as f:
    f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")

# Install tunnel
!pip install -q streamlit pyngrok
!npm install -g localtunnel 2>/dev/null

# Launch Streamlit in background
import subprocess
proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8502', '--server.headless', 'true'])

import time; time.sleep(5)

# Create tunnel
!npx localtunnel --port 8502

## 12. Download Results

In [ ]:
from google.colab import files
files.download('colab_evaluation_results.csv')
if os.path.exists('evaluation_charts.png'):
    files.download('evaluation_charts.png')
print('✅ Files downloaded!')